# Smoke ABSA QLoRA no Kaggle

Selecione **GPU T4** e habilite Internet em Settings. Confira o saldo de GPU da conta e preencha a primeira célula. Rode as células em ordem. O worker usa uma única T4 e tem limite de 25 minutos, incluindo instalação, download, 20 passos e recarga do adapter. O comando termina o processo, e o encerramento da sessão Kaggle é manual.

Amostra determinística de 100 IDs do treino. Batch 1 e acumulação 5. O teste de perda usa três exemplos dessa mesma amostra para verificar que a pilha aprende. A avaliação final usa os splits reservados posteriormente.

Modelo base público na revisão fixada. [Ficha e licença Apache 2.0](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).

Commit dos insumos usados no Kaggle: `1e25fbff62d9050391a2300eb82d5b6d3c5d932b`.
SHA-256 do código do worker: `298b6d8a0d09e4094511b3ae3186925aad65c203da6afe2791bc31ef2d8dab61`.


In [ ]:
from math import isfinite
from pathlib import Path
from uuid import uuid4

# Preencha com o saldo que o Kaggle mostra em Settings / Session management.
GPU_HOURS_REMAINING = None
if (type(GPU_HOURS_REMAINING) not in (float, int) or not isfinite(GPU_HOURS_REMAINING)
        or GPU_HOURS_REMAINING < 0.5):
    raise ValueError("Informe o saldo real de GPU em horas. Reserve pelo menos 0,5 h para o smoke.")
RUN = Path("/kaggle/working") / ("absa-qlora-smoke-" + uuid4().hex[:8])
RUN.mkdir(exist_ok=False)
print("Saída:", RUN)


In [ ]:
%%writefile /kaggle/working/absa_qlora_smoke_worker.py
import gc
import hashlib
import inspect
import json
import math
import os
import platform
import subprocess
import sys
import time
import traceback
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from statistics import mean, median
from urllib.request import urlopen

ROOT = Path(sys.argv[1])
QUOTA = float(sys.argv[2])
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
REV = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
TRAIN_SHA = "5a2db6e333e4055a387f49d3fc3ef019668b8d28157d1c340d73e20354b644c2"
SOURCE = "1e25fbff62d9050391a2300eb82d5b6d3c5d932b"
PINS = ("trl==1.17.1", "peft==0.21.2", "datasets==5.1.0")
RUNTIME = ("torch", "transformers", "bitsandbytes", "accelerate")
PACKAGES = (*RUNTIME, "trl", "peft", "datasets", "tokenizers", "huggingface-hub", "pyarrow", "dill")
STEPS, ACCUMULATION, SEED = 20, 5, 2718
GIB = 2**30
started = time.perf_counter()
report = dict(status="running", stage="start", model=BASE, revision=REV, source_commit=SOURCE,
              weekly_gpu_hours_remaining_at_start=QUOTA, python=platform.python_version(),
              smoke_examples=100, steps=STEPS, batch_size=1, gradient_accumulation=ACCUMULATION,
              max_length=256, seed=SEED, learning_rate=2e-4)


def versions(names):
    result = {}
    for name in names:
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = None
    return result


def record(stage, **values):
    report.update(stage=stage, elapsed_s=time.perf_counter() - started, **values)
    path = ROOT / "smoke.json"
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)
    print(f"stage={stage}", flush=True)


def main():
    runtime = versions(RUNTIME)
    record("preflight", packages_before=versions(PACKAGES))
    import torch

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Select a T4 accelerator. The worker must see exactly one CUDA device.")
    gpu = torch.cuda.get_device_properties(0)
    record("gpu_identified", gpu=gpu.name, cuda=torch.version.cuda,
           gpu_capability=[gpu.major, gpu.minor], gpu_total_gib=gpu.total_memory / GIB)
    expected = {"torch": "2.11.", "transformers": "5."}
    if "T4" not in gpu.name or torch.version.cuda != "13.0" or any(
        not (runtime[name] or "").startswith(prefix) for name, prefix in expected.items()
    ) or not runtime["accelerate"] or (
        runtime["bitsandbytes"] and not runtime["bitsandbytes"].startswith("0.50.")
    ):
        raise RuntimeError("Runtime differs from the targeted T4 / torch 2.11 / CUDA 13 / Transformers 5 / BNB 0.50 stack.")
    constraints = ROOT / "constraints.txt"
    constraints.write_text("\n".join(f"{name}=={ver}" for name, ver in runtime.items() if ver is not None) + "\n")
    pins = PINS + (("bitsandbytes==0.50.2",) if runtime["bitsandbytes"] is None else ())
    record("install", install_pins=pins)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "--constraint", str(constraints), *pins], check=True)
    record("import", packages=versions(PACKAGES))
    expected_runtime = {**runtime, "bitsandbytes": runtime["bitsandbytes"] or "0.50.2"}
    if versions(RUNTIME) != expected_runtime:
        raise RuntimeError("pip changed the runtime stack")
    import bitsandbytes as bnb
    from datasets import Dataset
    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback, set_seed,
    )
    from trl import SFTConfig, SFTTrainer

    if "processing_class" not in inspect.signature(SFTTrainer).parameters:
        raise RuntimeError("Unexpected installed SFTTrainer signature")
    required = {"max_length", "completion_only_loss", "assistant_only_loss"}
    if not required <= set(inspect.signature(SFTConfig).parameters):
        raise RuntimeError("Unexpected installed SFTConfig signature")
    set_seed(SEED)
    record("dataset")
    url = f"https://raw.githubusercontent.com/JoaoNordemann/partner-copilot/{SOURCE}/data/absa/train-v1.jsonl"
    with urlopen(url, timeout=30) as response:
        raw = response.read().replace(b"\r\n", b"\n")
    if hashlib.sha256(raw).hexdigest() != TRAIN_SHA:
        raise ValueError("Train dataset checksum mismatch")
    rows = list(map(json.loads, raw.decode("utf-8").splitlines()))
    selected = sorted(rows, key=lambda row: hashlib.sha256(f"{SEED}:{row['id']}".encode()).digest())[:100]
    if len({row["id"] for row in selected}) != 100:
        raise ValueError("Expected 100 distinct training IDs")
    examples = [{"prompt": row["messages"][:1], "completion": row["messages"][1:]} for row in selected]
    for item in examples:
        if [m["role"] for m in item["prompt"] + item["completion"]] != ["user", "assistant"]:
            raise ValueError("Invalid conversational roles")
    dataset = Dataset.from_generator(lambda: iter(examples), cache_dir=str(ROOT / "datasets-cache"))
    record("model_load", train_sha256=TRAIN_SHA, example_ids=[row["id"] for row in selected])
    tokenizer = AutoTokenizer.from_pretrained(BASE, revision=REV, token=False)
    config4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

    def load_base(local=False):
        return AutoModelForCausalLM.from_pretrained(
            BASE, revision=REV, token=False, local_files_only=local, device_map={"": 0},
            quantization_config=config4, dtype=torch.float16, attn_implementation="sdpa",
        )

    free, total = torch.cuda.mem_get_info(0)
    torch.cuda.reset_peak_memory_stats(0)
    model = load_base()
    if not getattr(model, "is_loaded_in_4bit", False) or not any(
        isinstance(layer, bnb.nn.Linear4bit) for layer in model.modules()
    ):
        raise RuntimeError("Base model did not load with BNB 4-bit layers")
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0,
                                          target_modules="all-linear", task_type="CAUSAL_LM",
                                          revision=REV))
    trainable = {name: param.numel() for name, param in model.named_parameters() if param.requires_grad}
    if not trainable or any("lora_" not in name for name in trainable):
        raise RuntimeError("Expected only LoRA parameters to be trainable")
    record("sft_init", trainable_parameters=sum(trainable.values()))

    class StepTimer(TrainerCallback):
        def __init__(self):
            self.seconds = []

        def on_step_begin(self, args, state, control, **kwargs):
            torch.cuda.synchronize()
            self.step_start = time.perf_counter()

        def on_step_end(self, args, state, control, **kwargs):
            torch.cuda.synchronize()
            self.seconds.append(time.perf_counter() - self.step_start)
            record("train", completed_steps=state.global_step, step_seconds=self.seconds)

    timer = StepTimer()
    args = SFTConfig(
        output_dir=str(ROOT / "trainer"), max_steps=STEPS, max_length=256,
        per_device_train_batch_size=1, gradient_accumulation_steps=ACCUMULATION,
        completion_only_loss=True, assistant_only_loss=False, packing=False,
        learning_rate=2e-4, lr_scheduler_type="constant", optim="adamw_torch",
        fp16=True, bf16=False, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, seed=SEED, data_seed=SEED,
        logging_steps=1, logging_nan_inf_filter=False, save_strategy="no", report_to="none",
        eos_token=tokenizer.eos_token,
    )
    trainer = SFTTrainer(model=model, args=args, processing_class=tokenizer,
                         train_dataset=dataset, callbacks=[timer])
    # TRL 1.17.1 casts trainable quantized adapters to BF16 at SFTTrainer init.
    # T4 FP16 GradScaler needs supported gradient dtypes. Keep LoRA masters FP32.
    adapters = [parameter for parameter in trainer.model.parameters() if parameter.requires_grad]
    initial_dtypes = sorted({str(parameter.dtype) for parameter in adapters})
    with torch.no_grad():
        for parameter in adapters:
            parameter.data = parameter.data.float()
    if not adapters or any(parameter.dtype != torch.float32 for parameter in adapters):
        raise RuntimeError("FP32 LoRA master weights were not applied after SFT initialization")
    record("completion_mask", adapter_dtypes_after_trl_init=initial_dtypes,
           trainable_adapter_dtype="torch.float32")
    probes = []
    for i, example in enumerate(examples):
        batch = trainer.data_collator([trainer.train_dataset[i]])
        ids = batch["input_ids"][0].tolist()
        labels = batch["labels"][0].tolist()
        prefix = tokenizer.apply_chat_template(example["prompt"], tokenize=True,
                                               add_generation_prompt=True, return_dict=True)["input_ids"]
        boundary = len(prefix)
        if not (0 < boundary < len(ids) <= 256 and ids[:boundary] == prefix
                and labels[:boundary] == [-100] * boundary and labels[boundary:] == ids[boundary:]):
            raise ValueError(f"Invalid completion mask or truncated sequence at example {i}")
        if tokenizer.decode(labels[boundary:], skip_special_tokens=True).strip() != example["completion"][0]["content"]:
            raise ValueError(f"Supervised target differs from canonical JSON at example {i}")
        if i < 3:
            probes.append(batch)
        if i == 0:
            trace = [{"position": n, "token_id": token, "label": label,
                      "token": tokenizer.convert_ids_to_tokens(token)}
                     for n, (token, label) in enumerate(zip(ids, labels, strict=True))]
            (ROOT / "mask-example.json").write_text(json.dumps(trace, ensure_ascii=False, indent=2), encoding="utf-8")
            print("prompt labels:", labels[:boundary])
            print("completion labels:", labels[boundary:])
            print("completion:", tokenizer.decode(labels[boundary:], skip_special_tokens=True).strip())
    record("probe_before", masks_verified=len(examples))

    def probe_loss(active_model):
        active_model.eval()
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            losses = [float(active_model(**{k: v.to("cuda:0") for k, v in batch.items()}).loss)
                      for batch in probes]
        if not all(math.isfinite(loss) for loss in losses):
            raise RuntimeError("Non-finite probe loss")
        return mean(losses)

    before = probe_loss(model)
    record("train", probe_loss_before=before)
    trainer.train()
    del adapters, parameter
    after = probe_loss(model)
    loss_curve = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry]
    if trainer.state.global_step != STEPS or len(timer.seconds) != STEPS or not loss_curve or not all(map(math.isfinite, loss_curve)):
        raise RuntimeError("Smoke did not finish 20 finite optimizer steps")
    peak_allocated = torch.cuda.max_memory_allocated(0)
    peak_reserved = torch.cuda.max_memory_reserved(0)
    headroom = (free - peak_reserved) / GIB
    steady = median(timer.seconds[2:])
    record("save", probe_loss_after=after, train_losses=loss_curve,
           peak_allocated_gib=peak_allocated / GIB, peak_reserved_gib=peak_reserved / GIB,
           gpu_total_gib=total / GIB, estimated_free_headroom_gib=headroom,
           median_seconds_per_optimizer_step=steady,
           estimated_train_minutes_per_epoch=math.ceil(len(rows) / ACCUMULATION) * steady / 60)
    adapter = ROOT / "adapter"
    model.save_pretrained(adapter, safe_serialization=True)
    tokenizer.save_pretrained(adapter)
    if not (adapter / "adapter_config.json").is_file() or not (adapter / "adapter_model.safetensors").is_file():
        raise RuntimeError("Adapter artifacts missing")
    trainer.accelerator.free_memory()
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    record("reload")
    fresh = prepare_model_for_kbit_training(load_base(local=True), use_gradient_checkpointing=False)
    reloaded = PeftModel.from_pretrained(fresh, adapter, is_trainable=False)
    restored = probe_loss(reloaded)
    checks = dict(loss_decreased=after < before, headroom_at_least_2_gib=headroom >= 2,
                  adapter_reloaded=math.isclose(after, restored, rel_tol=0.01, abs_tol=0.01))
    record("complete", status="passed" if all(checks.values()) else "failed_checks",
           checks=checks, probe_loss_reloaded=restored)
    return 0 if all(checks.values()) else 1


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except Exception as exc:
        record(report["stage"], status="error", packages=versions(PACKAGES),
               error_type=type(exc).__name__, error=str(exc), traceback=traceback.format_exc())
        raise


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

worker = Path("/kaggle/working/absa_qlora_smoke_worker.py")
log = RUN / "smoke.log"
with log.open("w", encoding="utf-8") as stream:
    proc = subprocess.run(
        ["timeout", "--signal=TERM", "--kill-after=5s", "25m", sys.executable, "-u",
         str(worker), str(RUN), str(GPU_HOURS_REMAINING)],
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "0", "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
             "TOKENIZERS_PARALLELISM": "false"},
        stdout=stream, stderr=subprocess.STDOUT,
    )
report_file = RUN / "smoke.json"
report = json.loads(report_file.read_text(encoding="utf-8")) if report_file.exists() else {}
report["worker_exit_code"] = proc.returncode
if proc.returncode in (124, 137):
    report["status"] = "time_limit"
    report["error"] = "25-minute worker limit reached or worker killed"
elif proc.returncode and report.get("status") in (None, "running"):
    report["status"] = "worker_error"
    report["error"] = f"Worker exited with code {proc.returncode}. See smoke.log."
report_file.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(log.read_text(encoding="utf-8", errors="replace")[-12000:])
print(json.dumps(report, ensure_ascii=False, indent=2))
bundle = RUN.parent / f"{RUN.name}.zip"
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    for name in ("smoke.json", "smoke.log", "mask-example.json", "constraints.txt"):
        artifact = RUN / name
        if artifact.is_file():
            archive.write(artifact, arcname=name)
    for artifact in sorted((RUN / "adapter").rglob("*")):
        if artifact.is_file():
            archive.write(artifact, arcname=artifact.relative_to(RUN))
print("Baixe o pacote de evidências e encerre a sessão GPU:", bundle)
if proc.returncode:
    raise RuntimeError(f"Smoke terminou com {report['status']}. Artefatos em {RUN}.")
